# Chain-of-Verification — Self-Correcting Factual Answers
## Claim, Check, Correct — Self-Verification Against Hallucination
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/68-chain-of-verification/chain_of_verification_workbook.ipynb)

LLMs hallucinate. **Chain-of-Verification (CoVe)** (Dhuliawala et al., 2023) is a structured self-correction technique: generate an initial answer, decompose it into verifiable claims, verify each claim independently, then revise only the failed ones. This workshop builds the full CoVe pipeline as a LangGraph graph. By the end you will understand *how* to make an LLM fact-check itself and *why* independent claim verification outperforms a single monolithic self-critique.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — How CoVe works and why independent verification matters |
| 2 | **Claim Extraction** — `ClaimList` Pydantic model |
| 3 | **Claim Verification** — `ClaimVerification` with correction field |
| 4 | **Revision** — Patch only failed claims into the answer |
| 5 | **Full Pipeline** — generate → plan → verify → revise |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "pydantic", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — How CoVe Works

```
Question
  │
  ▼
[generate]  → initial answer (may contain errors)
  │
  ▼
[plan_verification]  → extract verifiable claims from the answer
  │
  ▼
[execute_verification]  → verify each claim INDEPENDENTLY
  │                         (no memory of initial answer → reduces anchoring bias)
  ▼
[revise]  → patch failed claims into corrected answer
  │
  ▼
Final Answer
```

**Why independent?** If the verifier sees the original answer, it anchors to it. Asking "Is this claim true or false?" without showing the answer source forces genuine fact-checking.

In [ ]:
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# temperature=0 — deterministic outputs so verification judgments are consistent run-to-run
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

SAMPLE_QUESTIONS = [
    "What year was Albert Einstein born and what was his nationality?",
    "Who created the Python programming language and when was it first released?",
]
print(f"Demo questions: {len(SAMPLE_QUESTIONS)}")

---
## Part 2 — Claim Extraction with `ClaimList`

In [ ]:
# Pydantic model constrains the LLM to always return a list of strings — no freeform text to parse
class ClaimList(BaseModel):
    claims: list[str]

claim_extractor = llm.with_structured_output(ClaimList)
# with_structured_output wraps the LLM call in function-calling mode, validating against ClaimList

q = SAMPLE_QUESTIONS[0]
initial_answer = llm.invoke([
    SystemMessage(content="Answer factually and concisely."),
    HumanMessage(content=q)
]).content.strip()

print(f"Q: {q}")
print(f"Initial answer: {initial_answer}")

claims = claim_extractor.invoke([
    SystemMessage(content="Extract every distinct factual claim from this answer as a list of short, atomic statements."),
    HumanMessage(content=initial_answer)
])

print(f"\nExtracted {len(claims.claims)} claims:")
for i, c in enumerate(claims.claims, 1):
    print(f"  {i}. {c}")

---
## Part 3 — Independent Claim Verification

In [ ]:
# Atomic schema: one claim, one verdict, one correction — keeps verification focused and parseable
class ClaimVerification(BaseModel):
    claim: str
    correct: bool
    correction: str  # empty string if correct=True

claim_verifier = llm.with_structured_output(ClaimVerification)

verifications = []
# Sequential per-claim calls: slower but each claim gets an independent context window
for claim in claims.claims:
    v = claim_verifier.invoke([
        SystemMessage(content="You are a fact-checker. Verify this claim. If correct=False, provide the correction."),
        HumanMessage(content=claim)
    ])
    verifications.append(v)
    status = "✓" if v.correct else "✗"
    print(f"{status} {v.claim}")
    if not v.correct:
        print(f"    Correction: {v.correction}")

---
## Part 4 — Revision (Patch Failed Claims Only)

In [ ]:
# Only failed claims go into the revision prompt — avoids rewriting correct content unnecessarily
failed = [v for v in verifications if not v.correct]
print(f"Failed claims: {len(failed)} / {len(verifications)}")

if failed:
    corrections_text = "\n".join(f"- WRONG: {v.claim} → CORRECT: {v.correction}" for v in failed)
    revised = llm.invoke([
        SystemMessage(content="Revise the answer to fix only the incorrect claims listed below. Keep the rest unchanged."),
        HumanMessage(content=f"Original answer: {initial_answer}\n\nCorrections:\n{corrections_text}")
    ]).content.strip()
    print(f"\nOriginal: {initial_answer}")
    print(f"Revised:  {revised}")
else:
    print("All claims verified — no revision needed.")
    revised = initial_answer

---
## Part 5 — Full LangGraph Pipeline

```
START → generate → plan_verification → execute_verification → revise → END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

# TypedDict state: LangGraph passes this dict between nodes — each node returns only the keys it updates
class CoVeState(TypedDict):
    question: str; initial_answer: str
    claims: list; verifications: list; revised_answer: str

def generate_node(state):
    # Preserve a caller-supplied answer so the correction exercise can enter at START.
    if state["initial_answer"]:
        return {}
    ans = llm.invoke([
        SystemMessage(content="Answer factually and concisely."),
        HumanMessage(content=state["question"])
    ]).content.strip()
    return {"initial_answer": ans}

def plan_verification_node(state):
    cl = claim_extractor.invoke([
        SystemMessage(content="Extract every distinct factual claim as a list of short, atomic statements."),
        HumanMessage(content=state["initial_answer"])
    ])
    return {"claims": cl.claims}

def execute_verification_node(state):
    vlist = []
    for claim in state["claims"]:
        v = claim_verifier.invoke([
            SystemMessage(content="Fact-check this claim. If incorrect, provide the correction."),
            HumanMessage(content=claim)
        ])
        vlist.append(v.model_dump())
    return {"verifications": vlist}

def revise_node(state):
    failed = [v for v in state["verifications"] if not v["correct"]]
    if not failed:
        return {"revised_answer": state["initial_answer"]}
    corrections_text = "\n".join(f"- WRONG: {v['claim']} → CORRECT: {v['correction']}" for v in failed)
    revised = llm.invoke([
        SystemMessage(content="Revise the answer to fix only the incorrect claims. Keep the rest unchanged."),
        HumanMessage(content=f"Original: {state['initial_answer']}\n\nCorrections:\n{corrections_text}")
    ]).content.strip()
    return {"revised_answer": revised}

# StateGraph(CoVeState) enforces that node return dicts only contain valid CoVeState keys
g = StateGraph(CoVeState)
for name, fn in [("generate", generate_node), ("plan_verification", plan_verification_node),
                 ("execute_verification", execute_verification_node), ("revise", revise_node)]:
    g.add_node(name, fn)
g.add_edge(START, "generate")
g.add_edge("generate", "plan_verification")
g.add_edge("plan_verification", "execute_verification")
g.add_edge("execute_verification", "revise")
g.add_edge("revise", END)
# compile() locks the graph topology — after this point you can't add nodes or edges
app = g.compile()

for question in SAMPLE_QUESTIONS:
    init = {"question": question, "initial_answer": "", "claims": [], "verifications": [], "revised_answer": ""}
    r = app.invoke(init)
    failed_count = sum(1 for v in r["verifications"] if not v["correct"])
    print(f"Q: {question}")
    print(f"Claims: {len(r['claims'])}  |  Failed: {failed_count}")
    print(f"Initial: {r['initial_answer']}")
    print(f"Revised: {r['revised_answer']}")
    print()

---
### Exercise 1 — Inject a Deliberate Error
Replace `initial_answer` in state manually with a factually wrong answer and skip the `generate` node. Does the verification + revision pipeline correct it?

### Exercise 2 — Multiple Verification Passes
After revision, feed `revised_answer` back as the new question for a second CoVe pass. Does a second pass further improve accuracy, or does it introduce new errors?

In [ ]:
# Exercise 1 — inject a wrong initial answer
wrong_init = {
    "question": "What year was Albert Einstein born?",
    "initial_answer": "Albert Einstein was born in 1889 in France.",  # wrong year and country
    "claims": [], "verifications": [], "revised_answer": ""
}
# We still go through plan→verify→revise by jumping to plan_verification:
r = app.invoke({**wrong_init, "initial_answer": wrong_init["initial_answer"]})
print(f"Wrong input: {wrong_init['initial_answer']}")
print(f"Revised:     {r['revised_answer']}")

# Exercise 2 — second CoVe pass
# TODO: take r['revised_answer'] as initial_answer and run plan/verify/revise again

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*